# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Omesh-kumar/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Baseline Rule
This baseline rule identifies pages that are good candidates for content refresh. It gives higher scores to pages that have been stale for a long time, have low CTR, and still receive meaningful impressions.

Reason Code
STALE_LOW_CTR: The page is old, has low click-through rate, and should be reviewed for content refresh.
Action Label
REFRESH_CONTENT

In [21]:
import pandas as pd

df = pd.read_csv("/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv")

# Signal 1: CTR vs Position
df["position_bucket"] = pd.cut(
    df["avg_position"],
    bins=[0, 5, 10, 20, 50, 100],
    labels=["1-5", "6-10", "11-20", "21-50", "51+"]
)

signal1 = (
    df.groupby("position_bucket", observed=True)
      .agg(
          n=("content_id", "count"),
          avg_ctr=("ctr", "mean")
      )
      .reset_index()
)

print("Signal 1: CTR vs Position")
display(signal1)

Signal 1: CTR vs Position


,position_bucket,n,avg_ctr
0,1-5,3923,1.572937
1,6-10,9060,0.511708
2,11-20,7273,0.323443
3,21-50,7225,0.222345
4,51+,1299,0.152525


### Signal 1 Verdict: CONFIRMED

CTR decreases as average position becomes worse. Pages ranking in positions 1–5 have the highest average CTR (1.57), while pages beyond position 50 have the lowest CTR (0.15). This confirms the expected CTR vs Position relationship used by FlyRank.

In [22]:
# Signal 2: Impressions vs Search Volume

df["impression_bucket"] = pd.qcut(
    df["impressions_90d"],
    q=5,
    duplicates="drop"
)

signal2 = (
    df.groupby("impression_bucket", observed=True)
      .agg(
          n=("content_id", "count"),
          avg_search_volume=("search_volume", "mean")
      )
      .reset_index()
)

print("Signal 2: Impressions vs Search Volume")
display(signal2)

Signal 2: Impressions vs Search Volume


,impression_bucket,n,avg_search_volume
0,"(0.999, 39.0]",6041,131.131901
1,"(39.0, 364.0]",5964,153.166275
2,"(364.0, 1375.0]",5997,159.407624
3,"(1375.0, 5167.6]",5998,214.950864
4,"(5167.6, 517715.0]",6000,128.252101


### Signal 2 Verdict: MIXED

Higher impression buckets generally show higher average search volume, but the highest impression bucket does not have the highest average search volume. This suggests that search volume alone does not fully explain impressions, and other factors such as ranking position, CTR, or content quality also influence traffic.

In [23]:
import os

# Baseline rule
df["baseline_score"] = (
    (df["days_since_last_update"] / 30)
    + (100 - df["ctr"] * 100)
    + (df["impressions_90d"] / 1000)
)

df["reason_code"] = "STALE_LOW_CTR"
df["action_label"] = "REFRESH_CONTENT"

queue = (
    df.sort_values("baseline_score", ascending=False)
      [["content_id",
        "baseline_score",
        "reason_code",
        "action_label",
        "days_since_last_update",
        "ctr",
        "impressions_90d"]]
)

os.makedirs("/content/flyrank-ml-internship/work/outputs", exist_ok=True)

queue.to_csv(
    "/content/flyrank-ml-internship/work/outputs/baseline_action_score.csv",
    index=False
)

print(queue.head(10))
print("\nCSV saved successfully.")

                 content_id  baseline_score    reason_code     action_label  \
6653   content_5fe46e04994d      607.181667  STALE_LOW_CTR  REFRESH_CONTENT   
26844  content_8c19996aa890      594.918667  STALE_LOW_CTR  REFRESH_CONTENT   
17812  content_aaef01a50def      592.842333  STALE_LOW_CTR  REFRESH_CONTENT   
19636  content_2cb567c3c89b      589.327000  STALE_LOW_CTR  REFRESH_CONTENT   
29400  content_2dba2b1f9536      525.900667  STALE_LOW_CTR  REFRESH_CONTENT   
21819  content_4c36c775b818      522.769667  STALE_LOW_CTR  REFRESH_CONTENT   
29879  content_1a9e894be2e2      493.913333  STALE_LOW_CTR  REFRESH_CONTENT   
18870  content_db5989a78dd3      424.777667  STALE_LOW_CTR  REFRESH_CONTENT   
13537  content_2c2606c5d176      397.865667  STALE_LOW_CTR  REFRESH_CONTENT   
26531  content_cb112fce36be      397.376667  STALE_LOW_CTR  REFRESH_CONTENT   

       days_since_last_update   ctr  impressions_90d  
6653                      104  0.14           517715  
26844               

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [24]:
import os

df["baseline_score"] = (
    (df["days_since_last_update"] / 30)
    + (100 - df["ctr"] * 100)
    + (df["impressions_90d"] / 1000)
)

df["reason_code"] = "STALE_LOW_CTR"
df["action_label"] = "REFRESH_CONTENT"

queue = (
    df.sort_values("baseline_score", ascending=False)
      [["content_id",
        "baseline_score",
        "reason_code",
        "action_label",
        "days_since_last_update",
        "ctr",
        "impressions_90d"]]
)

os.makedirs("/content/flyrank-ml-internship/work/outputs", exist_ok=True)

queue.to_csv(
    "/content/flyrank-ml-internship/work/outputs/baseline_action_score.csv",
    index=False
)

print(queue.head(10))
print("CSV saved successfully.")

                 content_id  baseline_score    reason_code     action_label  \
6653   content_5fe46e04994d      607.181667  STALE_LOW_CTR  REFRESH_CONTENT   
26844  content_8c19996aa890      594.918667  STALE_LOW_CTR  REFRESH_CONTENT   
17812  content_aaef01a50def      592.842333  STALE_LOW_CTR  REFRESH_CONTENT   
19636  content_2cb567c3c89b      589.327000  STALE_LOW_CTR  REFRESH_CONTENT   
29400  content_2dba2b1f9536      525.900667  STALE_LOW_CTR  REFRESH_CONTENT   
21819  content_4c36c775b818      522.769667  STALE_LOW_CTR  REFRESH_CONTENT   
29879  content_1a9e894be2e2      493.913333  STALE_LOW_CTR  REFRESH_CONTENT   
18870  content_db5989a78dd3      424.777667  STALE_LOW_CTR  REFRESH_CONTENT   
13537  content_2c2606c5d176      397.865667  STALE_LOW_CTR  REFRESH_CONTENT   
26531  content_cb112fce36be      397.376667  STALE_LOW_CTR  REFRESH_CONTENT   

       days_since_last_update   ctr  impressions_90d  
6653                      104  0.14           517715  
26844               

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [25]:
top10 = queue.head(10)
display(top10)

,content_id,baseline_score,reason_code,action_label,days_since_last_update,ctr,impressions_90d
6653,content_5fe46e04994d,607.181667,STALE_LOW_CTR,REFRESH_CONTENT,104,0.14,517715
26844,content_8c19996aa890,594.918667,STALE_LOW_CTR,REFRESH_CONTENT,20,0.15,509252
17812,content_aaef01a50def,592.842333,STALE_LOW_CTR,REFRESH_CONTENT,22,0.25,517109
19636,content_2cb567c3c89b,589.327000,STALE_LOW_CTR,REFRESH_CONTENT,48,0.10,497727
29400,content_2dba2b1f9536,525.900667,STALE_LOW_CTR,REFRESH_CONTENT,104,0.21,443434
21819,content_4c36c775b818,522.769667,STALE_LOW_CTR,REFRESH_CONTENT,20,0.41,463103
29879,content_1a9e894be2e2,493.913333,STALE_LOW_CTR,REFRESH_CONTENT,22,0.23,416180
18870,content_db5989a78dd3,424.777667,STALE_LOW_CTR,REFRESH_CONTENT,20,0.21,345111
13537,content_2c2606c5d176,397.865667,STALE_LOW_CTR,REFRESH_CONTENT,104,0.53,347399
26531,content_cb112fce36be,397.376667,STALE_LOW_CTR,REFRESH_CONTENT,104,0.16,309910


## Top 10 Review

1.
Action: REFRESH_CONTENT
Reason Code: STALE_LOW_CTR
Why: High impressions, very low CTR, and stale content.
What would make it wrong: Traffic is seasonal.

2.
Action: REFRESH_CONTENT
Reason Code: STALE_LOW_CTR
Why: Strong visibility but poor CTR.
What would make it wrong: The page was recently redesigned.

3.
Action: REFRESH_CONTENT
Reason Code: STALE_LOW_CTR
Why: High impressions with low CTR.
What would make it wrong: Search intent has changed.

4.
Action: REFRESH_CONTENT
Reason Code: STALE_LOW_CTR
Why: Old content with many impressions.
What would make it wrong: Another page should replace it.

5.
Action: REFRESH_CONTENT
Reason Code: STALE_LOW_CTR
Why: High refresh opportunity.
What would make it wrong: Data quality issue.

6.
Action: REFRESH_CONTENT
Reason Code: STALE_LOW_CTR
Why: Low CTR despite high impressions.
What would make it wrong: An A/B title test is already running.

7.
Action: REFRESH_CONTENT
Reason Code: STALE_LOW_CTR
Why: High baseline score due to age and CTR.
What would make it wrong: Rankings recently improved.

8.
Action: REFRESH_CONTENT
Reason Code: STALE_LOW_CTR
Why: Large traffic opportunity.
What would make it wrong: The page is intentionally archived.

9.
Action: REFRESH_CONTENT
Reason Code: STALE_LOW_CTR
Why: Stale content with continued impressions.
What would make it wrong: Temporary traffic spike.

10.
Action: REFRESH_CONTENT
Reason Code: STALE_LOW_CTR
Why: Good refresh candidate.
What would make it wrong: Analytics data is incomplete.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Weak Picks

Some pages receive a high score because of only one strong signal. These pages should be manually reviewed before taking action.

### Leakage Check

No future information or label-derived features were used. The baseline only uses currently available metrics such as CTR, impressions, and days since last update.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

- ✅ Two signal checks completed.
- ✅ Bucket tables include n.
- ✅ One FlyRank signal verified.
- ✅ Baseline score implemented.
- ✅ Reason code added.
- ✅ Action label added.
- ✅ CSV generated successfully.
- ✅ Top review completed.
- ✅ No future leakage used.

In [26]:
from pathlib import Path
import pandas as pd

# Find dataset files
for p in Path("/content/flyrank-ml-internship").rglob("*"):
    if p.suffix in [".parquet", ".csv"]:
        print(p)

/content/flyrank-ml-internship/outputs/refresh_queue_sample.csv
/content/flyrank-ml-internship/flyrank-ml-internship/outputs/refresh_queue_sample.csv
/content/flyrank-ml-internship/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv
/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv
/content/flyrank-ml-internship/work/outputs/baseline_action_score.csv


In [27]:
import pandas as pd

df = pd.read_csv("/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv")

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

df.head()

Shape: (30000, 44)

Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [29]:
import os
print("Current:", os.getcwd())

Current: /content/flyrank-ml-internship/work/notebooks


In [30]:
!pwd

/content/flyrank-ml-internship/work/notebooks


In [31]:
!find /content -name w04_baseline_score.ipynb

/content/flyrank-ml-internship/flyrank-ml-internship/work/notebooks/w04_baseline_score.ipynb
/content/flyrank-ml-internship/work/notebooks/w04_baseline_score.ipynb


In [32]:
import os
print(os.getcwd())

/content/flyrank-ml-internship/work/notebooks


In [33]:
%cd /content/flyrank-ml-internship

/content/flyrank-ml-internship


In [34]:
!git status

On branch main
Your branch is up to date with 'origin/main'.

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	flyrank-ml-internship/

nothing added to commit but untracked files present (use "git add" to track)


In [35]:
!ls -la /content

total 20
drwxr-xr-x  1 root root 4096 Jul 19 08:28 .
drwxr-xr-x  1 root root 4096 Jul 19 08:21 ..
drwxr-xr-x  4 root root 4096 Jun  4 13:32 .config
drwxr-xr-x 13 root root 4096 Jul 19 08:29 flyrank-ml-internship
drwxr-xr-x  1 root root 4096 Jun  4 13:32 sample_data


In [36]:
!ls -la /content/flyrank-ml-internship

total 160
drwxr-xr-x 13 root root  4096 Jul 19 08:29 .
drwxr-xr-x  1 root root  4096 Jul 19 08:28 ..
-rw-r--r--  1 root root   654 Jul 19 08:28 AGENTS.md
-rw-r--r--  1 root root   654 Jul 19 08:28 CLAUDE.md
-rw-r--r--  1 root root 45547 Jul 19 08:28 Copy_of_w02_ml_task_framing.ipynb
drwxr-xr-x  3 root root  4096 Jul 19 08:28 data
-rw-r--r--  1 root root  2763 Jul 19 08:28 DATA_USE.md
drwxr-xr-x  2 root root  4096 Jul 19 08:28 docs
drwxr-xr-x 12 root root  4096 Jul 19 08:29 flyrank-ml-internship
drwxr-xr-x  8 root root  4096 Jul 19 09:03 .git
drwxr-xr-x  3 root root  4096 Jul 19 08:28 .github
-rw-r--r--  1 root root   993 Jul 19 08:28 .gitignore
-rw-r--r--  1 root root  9831 Jul 19 08:28 GUIDE.md
-rw-r--r--  1 root root  1289 Jul 19 08:28 LICENSE
drwxr-xr-x  2 root root  4096 Jul 19 08:28 notebooks
drwxr-xr-x  3 root root  4096 Jul 19 08:28 outputs
-rw-r--r--  1 root root  9561 Jul 19 08:28 README.md
-rw-r--r--  1 root root   107 Jul 19 08:28 requirements.txt
drwxr-xr-x  2 root root  40

In [37]:
!ls -l /content/flyrank-ml-internship/work/notebooks/w04_baseline_score.ipynb
!ls -l /content/flyrank-ml-internship/flyrank-ml-internship/work/notebooks/w04_baseline_score.ipynb

-rw-r--r-- 1 root root 2556 Jul 19 08:28 /content/flyrank-ml-internship/work/notebooks/w04_baseline_score.ipynb
-rw-r--r-- 1 root root 2556 Jul 19 08:29 /content/flyrank-ml-internship/flyrank-ml-internship/work/notebooks/w04_baseline_score.ipynb


In [39]:
!git -C /content/flyrank-ml-internship diff --name-only